# Case 4 — Geospatial Analytics untuk Digital Bank (Snowflake Notebook)

**Amar Bank — Workshop Part 2**

Use case: **Location Intelligence & Geo-Fraud** untuk digital bank.
- **A. Coverage** — seberapa jauh nasabah dari titik layanan (ATM/Cabang/Agen); temukan nasabah *underserved* & usulan lokasi baru.
- **B. Geo-Fraud** — deteksi *fraud ring* (banyak aplikasi dari titik sama) & *impossible travel* (satu nasabah, dua aplikasi berjauhan dalam waktu singkat).

Fungsi Snowflake native: `ST_MAKEPOINT`, `ST_DISTANCE`, `ST_CENTROID`, `H3_LATLNG_TO_CELL`, dll — tanpa library eksternal untuk komputasi. Visualisasi peta pakai **pydeck**.

**Tabel:** `CUSTOMERS_GEO` (10k), `SERVICE_POINTS` (200), `LOAN_APPLICATIONS_GEO` (15k).

**Packages:** `pandas numpy pydeck plotly` (aktifkan via tombol Packages).

In [ ]:
# Import & Snowpark session
import pandas as pd, numpy as np
import pydeck as pdk
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print('Session OK ->', session.get_current_database(), session.get_current_schema())

## 1. Muat data geospatial
Ambil 3 tabel ke pandas. `latitude`/`longitude` adalah titik lokasi (derajat desimal).

In [ ]:
cust = session.table('CUSTOMERS_GEO').to_pandas()
sp   = session.table('SERVICE_POINTS').to_pandas()
apps = session.table('LOAN_APPLICATIONS_GEO').to_pandas()
print('customers', cust.shape, '| service_points', sp.shape, '| applications', apps.shape)
cust.head()

## 2. Membuat titik GEOGRAPHY — `ST_MAKEPOINT`
`ST_MAKEPOINT(longitude, latitude)` membentuk objek GEOGRAPHY (titik di bumi).
**Perhatikan urutan: longitude dulu, baru latitude.** Titik ini dasar semua analisa jarak.

In [ ]:
demo = session.sql('''
  SELECT customer_id, city,
         ST_ASWKT(ST_MAKEPOINT(longitude, latitude)) AS geo_point
  FROM CUSTOMERS_GEO LIMIT 5
''').to_pandas()
demo

## 3. Coverage — jarak nasabah ke titik layanan TERDEKAT (`ST_DISTANCE`)
`ST_DISTANCE(g1, g2)` = jarak geodesik (meter) antara dua titik.
Kita CROSS JOIN nasabah × service point, ambil **MIN** jarak per nasabah = jarak ke layanan terdekat. Ini menjawab *"titik A (nasabah) ke titik B (layanan) berapa jauh?"*.

In [ ]:
coverage = session.sql('''
  WITH c AS (SELECT customer_id, city, latitude, longitude,
                    ST_MAKEPOINT(longitude, latitude) g FROM CUSTOMERS_GEO),
       s AS (SELECT service_point_id, ST_MAKEPOINT(longitude, latitude) g FROM SERVICE_POINTS)
  SELECT c.customer_id, c.city, c.latitude, c.longitude,
         MIN(ST_DISTANCE(c.g, s.g))/1000 AS nearest_km
  FROM c CROSS JOIN s
  GROUP BY 1,2,3,4
''').to_pandas()
coverage['UNDERSERVED'] = (coverage['NEAREST_KM'] > 5).astype(int)
print('Rata2 jarak ke layanan terdekat (km):', round(coverage.NEAREST_KM.mean(),2))
print('Nasabah underserved (>5km):', int(coverage.UNDERSERVED.sum()))
coverage.head()

### 3a. Coverage per kota
Kota dengan rata-rata jarak tinggi = kandidat penambahan ATM/cabang.

In [ ]:
import plotly.express as px
bycity = coverage.groupby('CITY', as_index=False)['NEAREST_KM'].mean().sort_values('NEAREST_KM', ascending=False)
px.bar(bycity, x='CITY', y='NEAREST_KM', color='NEAREST_KM', color_continuous_scale='Reds',
       title='Rata-rata jarak ke layanan terdekat per kota (km)')

## 4. Visualisasi peta — nasabah & titik layanan (pydeck)
Titik nasabah diwarnai: **merah = underserved (>5km)**, **biru = tercover**. Titik hijau = service point.
Ini memvisualisasikan lokasi titik A (nasabah) dan titik B (layanan).

In [ ]:
c = coverage.copy()
c['color'] = c['UNDERSERVED'].map(lambda u: [220,50,50] if u else [30,120,200])
cust_layer = pdk.Layer('ScatterplotLayer', data=c, get_position='[LONGITUDE, LATITUDE]',
                       get_fill_color='color', get_radius=250, opacity=0.5)
sp_layer = pdk.Layer('ScatterplotLayer', data=sp, get_position='[LONGITUDE, LATITUDE]',
                     get_fill_color='[20,180,90]', get_radius=600)
view = pdk.ViewState(latitude=-6.4, longitude=107.5, zoom=6)
st.pydeck_chart(pdk.Deck(layers=[cust_layer, sp_layer], initial_view_state=view,
                         map_style=None, tooltip={'text':'{CITY} - {NEAREST_KM} km'}))

## 5. Garis titik A → titik B (nasabah → layanan terdekat)
Untuk sampel nasabah underserved, kita tarik **garis** ke layanan terdekat memakai `LineLayer` — memperjelas jarak yang harus ditempuh.

In [ ]:
sample = session.sql('''
  WITH c AS (SELECT customer_id, latitude clat, longitude clon, ST_MAKEPOINT(longitude,latitude) g FROM CUSTOMERS_GEO),
       s AS (SELECT service_point_id, latitude slat, longitude slon, ST_MAKEPOINT(longitude,latitude) g FROM SERVICE_POINTS)
  SELECT customer_id, clat, clon, slat, slon, km FROM (
    SELECT c.customer_id, c.clat, c.clon, s.slat, s.slon,
           ST_DISTANCE(c.g,s.g)/1000 km,
           ROW_NUMBER() OVER (PARTITION BY c.customer_id ORDER BY ST_DISTANCE(c.g,s.g)) rn
    FROM c CROSS JOIN s )
  WHERE rn=1 AND km > 5
  ORDER BY km DESC LIMIT 200
''').to_pandas()
line_layer = pdk.Layer('LineLayer', data=sample, get_source_position='[CLON, CLAT]',
                       get_target_position='[SLON, SLAT]', get_color='[220,50,50]', get_width=2)
pts = pdk.Layer('ScatterplotLayer', data=sample, get_position='[CLON, CLAT]', get_fill_color='[220,50,50]', get_radius=300)
st.pydeck_chart(pdk.Deck(layers=[line_layer, pts], map_style=None,
                initial_view_state=pdk.ViewState(latitude=-6.4, longitude=107.5, zoom=6),
                tooltip={'text':'{KM} km ke layanan terdekat'}))

## 6. H3 Hexagon — sebaran & risiko default per area
**H3** membagi bumi jadi hexagon. `H3_LATLNG_TO_CELL(lat,lon,res)` memberi ID sel.
Kita agregasi jumlah nasabah & default rate per hexagon → heatmap risiko wilayah.

In [ ]:
h3 = session.sql('''
  SELECT H3_LATLNG_TO_CELL_STRING(latitude, longitude, 7) AS h3,
         COUNT(*) n_customers,
         ROUND(AVG(default_flag)*100,1) default_rate
  FROM CUSTOMERS_GEO GROUP BY 1
''').to_pandas()
print('jumlah hexagon:', len(h3))
layer = pdk.Layer('H3HexagonLayer', data=h3, get_hexagon='H3',
                  get_fill_color='[255, 140 - DEFAULT_RATE*2, 0, 160]',
                  get_elevation='N_CUSTOMERS', elevation_scale=20, extruded=True, pickable=True)
st.pydeck_chart(pdk.Deck(layers=[layer], map_style=None,
                initial_view_state=pdk.ViewState(latitude=-6.4, longitude=107.5, zoom=6, pitch=40),
                tooltip={'text':'cust={N_CUSTOMERS} default%={DEFAULT_RATE}'}))

## 7. Geo-Fraud (A) — *fraud ring*: banyak aplikasi dari titik sama
Aplikasi sah tersebar; **cluster titik nyaris identik** mencurigakan. Kita pakai H3 resolusi tinggi (res 9 ≈ 174m) dan hitung aplikasi per sel. Sel dengan aplikasi jauh di atas normal = kandidat fraud.

In [ ]:
cluster = session.sql('''
  SELECT H3_LATLNG_TO_CELL_STRING(latitude, longitude, 9) h3,
         COUNT(*) n_apps, COUNT(DISTINCT customer_id) n_cust
  FROM LOAN_APPLICATIONS_GEO GROUP BY 1
  ORDER BY n_apps DESC
''').to_pandas()
print('Top cluster (kandidat fraud ring):')
display(cluster.head(5))
thr = cluster.N_APPS.quantile(0.999)
print('Ambang p99.9 =', round(thr,1), '-> sel di atas ambang:', int((cluster.N_APPS>thr).sum()))

## 8. Geo-Fraud (B) — *impossible travel*
Satu nasabah mengajukan dari dua lokasi **berjauhan (>100 km)** dalam **selang < 3 jam** — mustahil ditempuh → indikasi akun disalahgunakan. `ST_DISTANCE` + `LAG` window.
> Catatan: `LAG` tidak menerima GEOGRAPHY, jadi kita LAG lat/long lalu bentuk titik.

In [ ]:
travel = session.sql('''
  WITH a AS (
    SELECT customer_id, submit_ts::timestamp ts, latitude, longitude,
           LAG(submit_ts::timestamp) OVER (PARTITION BY customer_id ORDER BY submit_ts) prev_ts,
           LAG(latitude)  OVER (PARTITION BY customer_id ORDER BY submit_ts) prev_lat,
           LAG(longitude) OVER (PARTITION BY customer_id ORDER BY submit_ts) prev_lon
    FROM LOAN_APPLICATIONS_GEO)
  SELECT customer_id, prev_ts, ts,
         ROUND(ST_DISTANCE(ST_MAKEPOINT(longitude,latitude), ST_MAKEPOINT(prev_lon,prev_lat))/1000,1) dist_km,
         DATEDIFF('minute', prev_ts, ts) mins,
         latitude, longitude, prev_lat, prev_lon
  FROM a
  WHERE prev_ts IS NOT NULL
    AND ST_DISTANCE(ST_MAKEPOINT(longitude,latitude), ST_MAKEPOINT(prev_lon,prev_lat))/1000 > 100
    AND DATEDIFF('minute', prev_ts, ts) < 180
''').to_pandas()
print('Kasus impossible travel terdeteksi:', len(travel))
travel.head()

In [ ]:
if len(travel):
    tl = pdk.Layer('LineLayer', data=travel, get_source_position='[PREV_LON, PREV_LAT]',
                   get_target_position='[LONGITUDE, LATITUDE]', get_color='[230,30,30]', get_width=3)
    st.pydeck_chart(pdk.Deck(layers=[tl], map_style=None,
        initial_view_state=pdk.ViewState(latitude=-3.0, longitude=104.0, zoom=4),
        tooltip={'text':'cust {CUSTOMER_ID}: {DIST_KM} km dalam {MINS} menit'}))
else:
    print('Tidak ada kasus (coba longgarkan ambang).')

## 9. Simpan hasil ke Snowflake
Simpan flag underserved & kasus fraud untuk downstream (dashboard / alert).

In [ ]:
session.write_pandas(coverage, 'CUSTOMER_COVERAGE', auto_create_table=True, overwrite=True)
if len(travel):
    session.write_pandas(travel, 'GEO_FRAUD_IMPOSSIBLE_TRAVEL', auto_create_table=True, overwrite=True)
print('Tersimpan: CUSTOMER_COVERAGE', len(coverage), '| impossible travel', len(travel))

## 10. Kesimpulan
- **Coverage:** identifikasi nasabah jauh dari layanan → optimasi jaringan ATM/cabang/agen.
- **H3 risk map:** wilayah dengan konsentrasi & default tinggi.
- **Geo-fraud:** *fraud ring* (cluster titik) & *impossible travel* (jarak vs waktu).

Semua memakai fungsi geospatial **native Snowflake** (`ST_*`, `H3_*`) + visualisasi **pydeck**.

✅ Selesai — Case 4.